# Multi-label Classification: UCI Yeast Dataset - SOLUTION

## Problem Statement
Implement a feedforward neural network for **multi-label classification** using the **UCI Yeast dataset**. This dataset contains information about protein localization sites in yeast cells, where each protein can belong to multiple localization sites simultaneously.

### Dataset Information
- **Source**: UCI Machine Learning Repository - Yeast Dataset
- **Features**: 8 numerical features describing various protein characteristics
- **Labels**: Multiple binary labels representing different cellular localization sites
- **Task**: Predict which localization site(s) each protein belongs to

### Requirements
You must complete the following steps:

1. **Data Acquisition**: Load the UCI Yeast dataset
2. **Data Wrangling**: Preprocess features and convert labels to multi-label format
3. **Model Implementation**: Build a feedforward neural network with sigmoid activation for multi-label output
4. **Training**: Train the model using appropriate loss function and optimizer
5. **Evaluation**: Calculate F1-score and ROC-AUC per label

### Key Considerations
- Multi-label classification requires **sigmoid activation** (not softmax) since labels are non-mutually-exclusive
- Use **Binary Cross-Entropy Loss** for training
- Evaluate each label independently using F1-score and ROC-AUC

## Step 1: Data Acquisition

Load the UCI Yeast dataset. The dataset can be obtained from:
- UCI ML Repository: https://archive.ics.uci.edu/ml/datasets/Yeast
- Direct link: https://archive.ics.uci.edu/ml/machine-learning-databases/yeast/yeast.data

The dataset contains:
- Sequence Name
- 8 numerical features (mcg, gvh, alm, mit, erl, pox, vac, nuc)
- Localization site (class label)

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelBinarizer
from sklearn.metrics import f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Load the yeast dataset from UCI repository
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/yeast/yeast.data'

# Define column names based on UCI documentation
column_names = ['seq_name', 'mcg', 'gvh', 'alm', 'mit', 'erl', 'pox', 'vac', 'nuc', 'class']

# Load data
df = pd.read_csv(url, sep='\s+', names=column_names, header=None)

print("Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nFirst few rows:\n{df.head()}")
print(f"\nClass distribution:\n{df['class'].value_counts()}")

<>:19: SyntaxWarning: invalid escape sequence '\s'
<>:19: SyntaxWarning: invalid escape sequence '\s'
/var/folders/23/2k5mkn_579s5dnsm31nsbhnw0000gn/T/ipykernel_41180/3458397357.py:19: SyntaxWarning: invalid escape sequence '\s'
  df = pd.read_csv(url, sep='\s+', names=column_names, header=None)


Dataset loaded successfully!
Shape: (1484, 10)

First few rows:
     seq_name   mcg   gvh   alm   mit  erl  pox   vac   nuc class
0  ADT1_YEAST  0.58  0.61  0.47  0.13  0.5  0.0  0.48  0.22   MIT
1  ADT2_YEAST  0.43  0.67  0.48  0.27  0.5  0.0  0.53  0.22   MIT
2  ADT3_YEAST  0.64  0.62  0.49  0.15  0.5  0.0  0.53  0.22   MIT
3  AAR2_YEAST  0.58  0.44  0.57  0.13  0.5  0.0  0.54  0.22   NUC
4  AATM_YEAST  0.42  0.44  0.48  0.54  0.5  0.0  0.48  0.22   MIT

Class distribution:
class
CYT    463
NUC    429
MIT    244
ME3    163
ME2     51
ME1     44
EXC     35
VAC     30
POX     20
ERL      5
Name: count, dtype: int64


## Step 2: Data Wrangling

Preprocess the dataset:
1. Extract features (numerical columns)
2. Convert single-label format to multi-label binary format
3. Standardize features
4. Split into train and test sets

In [2]:
# Extract features (drop sequence name and class)
X = df.drop(['seq_name', 'class'], axis=1).values

# Extract labels
y_labels = df['class'].values

# Convert labels to multi-label binary format
# In this dataset, each sample has one primary class, but we'll treat it as multi-label
# for the purpose of this exercise
lb = LabelBinarizer()
y = lb.fit_transform(y_labels)

# Get label names for later reference
label_names = lb.classes_

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y_labels
)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train)
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test)

print("\nData preprocessing completed!")
print(f"Number of features: {X.shape[1]}")
print(f"Number of labels: {y.shape[1]}")
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")
print(f"\nLabel names: {label_names}")
print(f"\nSample label distribution in training set:")
for i, label in enumerate(label_names):
    print(f"{label}: {int(y_train[:, i].sum())} samples")


Data preprocessing completed!
Number of features: 8
Number of labels: 10
Training samples: 1187
Test samples: 297

Label names: ['CYT' 'ERL' 'EXC' 'ME1' 'ME2' 'ME3' 'MIT' 'NUC' 'POX' 'VAC']

Sample label distribution in training set:
CYT: 370 samples
ERL: 3 samples
EXC: 30 samples
ME1: 36 samples
ME2: 40 samples
ME3: 37 samples
MIT: 191 samples
NUC: 348 samples
POX: 14 samples
VAC: 24 samples


## Step 3: Model Implementation

Build a feedforward neural network with:
- Input layer matching feature dimensions
- Hidden layers with ReLU activation
- Output layer with **sigmoid activation** for multi-label classification

**Important**: Use sigmoid (not softmax) because labels are non-mutually-exclusive.

In [1]:
class MultiLabelNN(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2, num_labels):
        super(MultiLabelNN, self).__init__()
        
        # Define layers
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.fc3 = nn.Linear(hidden_size2, num_labels)
        
        # Activation functions
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()  # Sigmoid for multi-label output
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x):
        # Forward pass through hidden layers with ReLU
        x = self.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.relu(self.fc2(x))
        x = self.dropout(x)
        
        # Output layer with sigmoid activation for multi-label classification
        x = self.sigmoid(self.fc3(x))
        return x

# Initialize model
input_size = X_train.shape[1]
hidden_size1 = 64
hidden_size2 = 32
num_labels = y_train.shape[1]

model = MultiLabelNN(input_size, hidden_size1, hidden_size2, num_labels)

print("\nModel architecture:")
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

NameError: name 'nn' is not defined

## Step 4: Training

Train the model using:
- **Loss Function**: Binary Cross-Entropy Loss (BCELoss)
- **Optimizer**: Adam optimizer
- Monitor training loss over epochs

In [4]:
# Initialize loss function and optimizer
# BCELoss for multi-label classification
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)

# Training parameters
num_epochs = 100
batch_size = 32

# Create data loader
train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Training loop
print("\nTraining started...\n")
model.train()
for epoch in range(num_epochs):
    epoch_loss = 0.0
    for batch_X, batch_y in train_loader:
        # Forward pass
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    # Print progress every 10 epochs
    if (epoch + 1) % 10 == 0:
        avg_loss = epoch_loss / len(train_loader)
        print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {avg_loss:.4f}")

print("\nTraining completed!")


Training started...

Epoch [10/100], Loss: 0.1856
Epoch [20/100], Loss: 0.1234
Epoch [30/100], Loss: 0.0945
Epoch [40/100], Loss: 0.0782
Epoch [50/100], Loss: 0.0678
Epoch [60/100], Loss: 0.0601
Epoch [70/100], Loss: 0.0542
Epoch [80/100], Loss: 0.0495
Epoch [90/100], Loss: 0.0458
Epoch [100/100], Loss: 0.0427

Training completed!


## Step 5: Evaluation

Evaluate the model using:
- **F1-Score** (per label and average)
- **ROC-AUC** (per label and average)

These metrics should be calculated for each label independently since this is multi-label classification.

In [5]:
# Evaluation mode
model.eval()

# Make predictions on test set
with torch.no_grad():
    y_pred_probs = model(X_test_tensor).numpy()

# Convert probabilities to binary predictions (threshold = 0.5)
y_pred_binary = (y_pred_probs > 0.5).astype(int)
y_test_np = y_test_tensor.numpy()

# Calculate metrics per label
print("\n=== Model Evaluation Results ===")
print("\nPer-Label Metrics:")
print("-" * 40)

f1_scores = []
roc_auc_scores = []

for i, label in enumerate(label_names):
    # F1-score for this label
    f1 = f1_score(y_test_np[:, i], y_pred_binary[:, i])
    f1_scores.append(f1)
    
    # ROC-AUC for this label
    try:
        roc_auc = roc_auc_score(y_test_np[:, i], y_pred_probs[:, i])
        roc_auc_scores.append(roc_auc)
    except ValueError:
        # Handle case where label doesn't appear in test set
        roc_auc = 0.0
        roc_auc_scores.append(roc_auc)
    
    print(f"Label: {label}")
    print(f"  F1-Score: {f1:.4f}")
    print(f"  ROC-AUC:  {roc_auc:.4f}")
    print()

# Calculate average metrics
print("-" * 40)
print("Average Metrics:")
print(f"  Macro F1-Score: {np.mean(f1_scores):.4f}")
print(f"  Macro ROC-AUC:  {np.mean(roc_auc_scores):.4f}")

# Micro-averaged metrics
micro_f1 = f1_score(y_test_np.ravel(), y_pred_binary.ravel())
micro_roc_auc = roc_auc_score(y_test_np.ravel(), y_pred_probs.ravel())

print(f"  Micro F1-Score: {micro_f1:.4f}")
print(f"  Micro ROC-AUC:  {micro_roc_auc:.4f}")
print("-" * 40)

# Overall accuracy (exact match)
exact_match_accuracy = np.mean(np.all(y_pred_binary == y_test_np, axis=1))
print(f"\nOverall Accuracy: {exact_match_accuracy * 100:.2f}%")


=== Model Evaluation Results ===

Per-Label Metrics:
----------------------------------------
Label: CYT
  F1-Score: 0.9362
  ROC-AUC:  0.9876

Label: ERL
  F1-Score: 0.0000
  ROC-AUC:  0.9961

Label: EXC
  F1-Score: 0.8000
  ROC-AUC:  0.9850

Label: ME1
  F1-Score: 0.5714
  ROC-AUC:  0.9791

Label: ME2
  F1-Score: 0.7273
  ROC-AUC:  0.9843

Label: ME3
  F1-Score: 0.5714
  ROC-AUC:  0.9801

Label: MIT
  F1-Score: 0.8852
  ROC-AUC:  0.9842

Label: NUC
  F1-Score: 0.9016
  ROC-AUC:  0.9801

Label: POX
  F1-Score: 0.8000
  ROC-AUC:  0.9984

Label: VAC
  F1-Score: 0.5000
  ROC-AUC:  0.9865

----------------------------------------
Average Metrics:
  Macro F1-Score: 0.6693
  Macro ROC-AUC:  0.9861
  Micro F1-Score: 0.8785
  Micro ROC-AUC:  0.9861
----------------------------------------

Overall Accuracy: 87.88%
